# RecruitAI evaluation

Goal: see if the system picks the right action on the labeled SMS turns.

Labels are only on recruiter turns: `continue`, `schedule`, `end`.

I use `accuracy_score` and a confusion matrix heatmap (Lesson 17).

**Notes before running:**
- needs `OPENAI_API_KEY` (Exit Advisor is prompt-based, no saved model)
- the SMS dates are from 2024, but our schedule DB starts from today, so I evaluate as if the chat is happening *now*
- Monday is not in the seed calendar, so some schedule cases can still look weird

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

# notebook lives in tests/, so go one level up to the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.modules.agents import exit_advisor
from app.modules.agents.orchestrator import predict_action
from app.modules.evaluation.prepare_data import build_labeled_rows

df = build_labeled_rows()
print(df.shape)
print(df["label"].value_counts())
df.head()

## 1) Exit Advisor alone

First I only check Exit Advisor.

It returns `end` or `dont_end`, but the dataset has 3 labels, so I map:
- true label `end` -> `end`
- everything else (`continue`, `schedule`) -> `not_end`

In [ ]:
exit_preds = [exit_advisor.predict(text)["decision"] for text in df["history"]]

y_true_exit = ["end" if label == "end" else "not_end" for label in df["label"]]
y_pred_exit = ["end" if pred == "end" else "not_end" for pred in exit_preds]

print("Exit accuracy:", accuracy_score(y_true_exit, y_pred_exit))

cm_exit = confusion_matrix(y_true_exit, y_pred_exit, labels=["end", "not_end"])
sns.heatmap(
    cm_exit,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["end", "not_end"],
    yticklabels=["end", "not_end"],
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Exit Advisor")
plt.show()

## 2) Full action (Exit + Sched)

`predict_action` does what we need for scoring:
1. ask Exit Advisor
2. if not end, ask Sched Advisor
3. priority is end > schedule > continue

Info Advisor is skipped here because info still maps to continue anyway.

In [ ]:
# important: do NOT use the 2024 timestamps from the json
# otherwise Sched Advisor looks in the past and finds nothing
now = datetime.now()

predictions = []
for row in df.itertuples(index=False):
    result = predict_action(row.history, conversation_dt=now)
    predictions.append(result["action"])
    print(row.conversation_id, row.turn_id, row.label, "->", result["action"])

df = df.copy()
df["predicted"] = predictions

print("\nAccuracy:", accuracy_score(df["label"], df["predicted"]))
df[["conversation_id", "turn_id", "label", "predicted"]].head(10)

In [ ]:
labels = ["continue", "schedule", "end"]
cm = confusion_matrix(df["label"], df["predicted"], labels=labels)
print(cm)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("continue / schedule / end")
plt.show()

## What I noticed

- only recruiter turns have labels
- classes are not balanced (more continue than end)
- we score the **action**, not whether the SMS text matches the dataset
- if accuracy is low on schedule, check the calendar seed / Monday issue first
- this notebook is expensive to re-run because each row hits the API